# StreamingAttention — Sprint 2: Calibration

**Goal**: Two-stage calibration to close the perplexity gap from Sprint 1 zero-shot failure.

- **Stage 1**: Per-head MSE alignment — learn optimal decay λ per streaming head (teacher = softmax, student = state)
- **Stage 2**: End-to-end LoRA fine-tuning — cross-entropy LM loss with rank-16 adapters

**Target**: Perplexity within 5% of baseline (≈6.43) → < 6.75

---

## Cell 1: Environment Setup

In [ ]:
# === Colab Setup ===
!pip install -q transformers datasets peft accelerate matplotlib tqdm

import os

# Clone the repo (streaming_attention-calibration branch)
if not os.path.exists('PROJECT'):
    !git clone -b streaming_attention-calibration https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git

os.chdir('PROJECT')
!pip install -e ".[train]" -q

# HuggingFace login (required for gated models like Llama)
from huggingface_hub import login
from google.colab import userdata
try:
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    print('Set HF_TOKEN in Colab secrets, or run: login(token="hf_...")')

# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/streaming_attention_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Verify streaming_attention is importable
import streaming_attention
print(f"streaming_attention v{streaming_attention.__version__} loaded successfully")

import json
import logging
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('streaming_attention')

# Check GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.reset_peak_memory_stats()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## Cell 2: Load Model + Baseline Perplexity

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model: {MODEL_NAME} (bfloat16)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',  # Need eager to intercept forward pass
)
model.eval()

cfg = model.config
print(f"\nModel config:")
print(f"  Layers: {cfg.num_hidden_layers}")
print(f"  Q heads: {cfg.num_attention_heads}")
print(f"  KV heads: {cfg.num_key_value_heads}")
print(f"  Head dim: {cfg.hidden_size // cfg.num_attention_heads}")
print(f"  GQA ratio: {cfg.num_attention_heads // cfg.num_key_value_heads} Q per KV head")

# Baseline perplexity
from streaming_attention.eval_perplexity import evaluate_perplexity

print("\n" + "="*60)
print("BASELINE: Full KV cache perplexity on WikiText-2")
print("="*60)

baseline_result = evaluate_perplexity(
    model=model,
    tokenizer=tokenizer,
    dataset_name='wikitext',
    dataset_config='wikitext-2-raw-v1',
    split='test',
    max_length=2048,
    stride=512,
    device=DEVICE,
)

BASELINE_PPL = baseline_result['perplexity']
print(f"\n>>> Baseline perplexity: {BASELINE_PPL:.2f}")

## Cell 3: Load Head Classification (DuoAttention)

In [ ]:
from streaming_attention.head_classifier import load_duo_attention_patterns

STREAMING_FRACTION = 0.5

if not os.path.exists('duo-attention'):
    print("Cloning DuoAttention for pre-computed head patterns...")
    !git clone --depth 1 https://github.com/mit-han-lab/duo-attention.git duo-attention

DUO_ATTENTION_DIR = 'duo-attention/attn_patterns/Meta-Llama-3.1-8B-Instruct'

head_classification = load_duo_attention_patterns(
    DUO_ATTENTION_DIR,
    sparsity=STREAMING_FRACTION,
)

print(f"\nMask shape: {head_classification.mask.shape}")
print(f"Retrieval heads: {head_classification.num_retrieval}")
print(f"Streaming heads: {head_classification.num_streaming}")
print(f"Streaming fraction: {head_classification.streaming_fraction:.1%}")

## Cell 4: Patch Model with Learnable Decay

In [ ]:
from streaming_attention.hybrid_attention import patch_model_for_streaming_attention, StreamingAttentionConfig

state_config = StreamingAttentionConfig(
    decay_init=0.99,
    learnable_decay=True,      # Enable gradient flow through decay
    chunk_size=64,
    skip_feature_map=False,    # Keep elu+1 (needed for stable normalization)
)

print("Applying StreamingAttention patch for calibration...")
print(f"  Decay init: {state_config.decay_init}")
print(f"  Learnable decay: {state_config.learnable_decay}")
print(f"  Skip feature map: {state_config.skip_feature_map}")
print(f"  Chunk size: {state_config.chunk_size}")

model, state_cache = patch_model_for_streaming_attention(
    model=model,
    head_classification=head_classification,
    config=state_config,
)

print(f"\nPatch applied. State modules: {len(model._streaming_attn_modules)}")

# Verify decay params are learnable
decay_params = [(n, p) for n, p in model._streaming_attn_modules.named_parameters() if 'log_decay' in n]
print(f"Learnable decay parameters: {len(decay_params)}")
for name, p in decay_params[:5]:
    print(f"  {name}: decay={torch.sigmoid(p.data).item():.4f}, requires_grad={p.requires_grad}")
if len(decay_params) > 5:
    print(f"  ... ({len(decay_params) - 5} more)")

if torch.cuda.is_available():
    print(f"\nGPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Cell 5: Stage 1 — Per-Head MSE Alignment

In [ ]:
from streaming_attention.calibration import calibrate_stage1

print("="*60)
print("STAGE 1: Per-head MSE alignment (learning decay)")
print("="*60)

stage1_result = calibrate_stage1(
    model=model,
    tokenizer=tokenizer,
    head_classification=head_classification,
    state_cache=state_cache,
    dataset_name='DKYoon/SlimPajama-6B',
    lr=0.02,
    num_steps=500,
    batch_size=2,
    seq_len=128,
    warmup_steps=50,
    log_interval=50,
    device=DEVICE,
)

# Plot Stage 1 loss curve
plt.figure(figsize=(10, 4))
plt.plot(stage1_result['losses'], alpha=0.3, color='blue')
window = 20
if len(stage1_result['losses']) > window:
    smoothed = np.convolve(stage1_result['losses'], np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(stage1_result['losses'])), smoothed, color='blue', linewidth=2)
plt.xlabel('Step')
plt.ylabel('MSE Loss')
plt.title('Stage 1: Per-Head MSE Alignment Loss')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('stage1_loss.png', dpi=150)
plt.show()

# Print learned decays
print("\nLearned decay values:")
for name, val in sorted(stage1_result['final_decays'].items()):
    print(f"  {name}: {val:.4f}")

## Cell 6: Perplexity After Stage 1

In [ ]:
print("="*60)
print("POST-STAGE 1: Perplexity after decay calibration")
print("="*60)

state_cache.reset()
model.eval()

stage1_ppl_result = evaluate_perplexity(
    model=model,
    tokenizer=tokenizer,
    dataset_name='wikitext',
    dataset_config='wikitext-2-raw-v1',
    split='test',
    max_length=2048,
    stride=512,
    device=DEVICE,
)

stage1_ratio = stage1_ppl_result['perplexity'] / BASELINE_PPL

print(f"\n{'='*60}")
print(f"RESULTS AFTER STAGE 1")
print(f"{'='*60}")
print(f"Baseline perplexity:      {BASELINE_PPL:.2f}")
print(f"Post-Stage 1 perplexity:  {stage1_ppl_result['perplexity']:.2f}")
print(f"Degradation ratio:        {stage1_ratio:.2f}x")
print()
if stage1_ratio < 2.0:
    print(">>> Stage 1 target met (< 2x baseline). Proceeding to Stage 2.")
elif stage1_ratio < 5.0:
    print(">>> Stage 1 partial success. Stage 2 LoRA should help further.")
else:
    print(">>> Stage 1 insufficient. Consider more steps or different approach.")

## Cell 7: Stage 2 — LoRA Fine-Tuning

In [ ]:
from streaming_attention.calibration import calibrate_stage2

print("="*60)
print("STAGE 2: End-to-end LoRA fine-tuning")
print("="*60)

stage2_result = calibrate_stage2(
    model=model,
    tokenizer=tokenizer,
    state_cache=state_cache,
    lora_rank=16,
    lora_alpha=32,
    lora_target_modules=['q_proj', 'v_proj'],
    dataset_name='DKYoon/SlimPajama-6B',
    lr=2e-4,
    num_steps=2000,
    batch_size=2,
    seq_len=256,
    warmup_steps=100,
    log_interval=100,
    device=DEVICE,
)

# The model is now PEFT-wrapped
model = stage2_result['model']

# Plot Stage 2 loss curve
plt.figure(figsize=(10, 4))
plt.plot(stage2_result['losses'], alpha=0.3, color='red')
window = 50
if len(stage2_result['losses']) > window:
    smoothed = np.convolve(stage2_result['losses'], np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(stage2_result['losses'])), smoothed, color='red', linewidth=2)
plt.xlabel('Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('Stage 2: LoRA Fine-Tuning Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('stage2_loss.png', dpi=150)
plt.show()

## Cell 8: Final Perplexity

In [ ]:
print("="*60)
print("FINAL: Perplexity after Stage 1 + Stage 2")
print("="*60)

state_cache.reset()
model.eval()

final_result = evaluate_perplexity(
    model=model,
    tokenizer=tokenizer,
    dataset_name='wikitext',
    dataset_config='wikitext-2-raw-v1',
    split='test',
    max_length=2048,
    stride=512,
    device=DEVICE,
)

final_ratio = final_result['perplexity'] / BASELINE_PPL

print(f"\n{'='*60}")
print(f"FINAL RESULTS")
print(f"{'='*60}")
print(f"Baseline perplexity:       {BASELINE_PPL:.2f}")
print(f"Post-Stage 1 perplexity:   {stage1_ppl_result['perplexity']:.2f} ({stage1_ratio:.2f}x)")
print(f"Final perplexity:          {final_result['perplexity']:.2f} ({final_ratio:.2f}x)")
print(f"Target:                    < {BASELINE_PPL * 1.05:.2f} (1.05x baseline)")
print()
if final_ratio <= 1.05:
    print(">>> SUCCESS: Final perplexity within 5% of baseline!")
elif final_ratio <= 1.10:
    print(">>> CLOSE: Final perplexity within 10% of baseline. Consider more LoRA steps.")
elif final_ratio <= 2.0:
    print(">>> PARTIAL: Significant improvement but still > 10% gap. Needs more tuning.")
else:
    print(">>> INSUFFICIENT: Still > 2x baseline. Revisit approach.")

# Combined loss plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(stage1_result['losses'], alpha=0.3, color='blue')
if len(stage1_result['losses']) > 20:
    s = np.convolve(stage1_result['losses'], np.ones(20)/20, mode='valid')
    ax1.plot(range(19, len(stage1_result['losses'])), s, color='blue', linewidth=2)
ax1.set_xlabel('Step')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Stage 1: Decay Alignment')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

ax2.plot(stage2_result['losses'], alpha=0.3, color='red')
if len(stage2_result['losses']) > 50:
    s = np.convolve(stage2_result['losses'], np.ones(50)/50, mode='valid')
    ax2.plot(range(49, len(stage2_result['losses'])), s, color='red', linewidth=2)
ax2.set_xlabel('Step')
ax2.set_ylabel('CE Loss')
ax2.set_title('Stage 2: LoRA Fine-Tuning')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('calibration_losses.png', dpi=150)
plt.show()

## Cell 9: Save Checkpoint to Drive

In [ ]:
import shutil

SAVE_DIR = os.path.join(CHECKPOINT_DIR, 'sprint2_calibrated')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Saving checkpoint to: {SAVE_DIR}")

# 1. Save calibrated decay parameters
decay_dir = os.path.join(SAVE_DIR, 'decay_params')
os.makedirs(decay_dir, exist_ok=True)

# After Stage 2, model is PEFT-wrapped; access base model's streaming_attention modules
base = model.base_model.model if hasattr(model, 'base_model') else model
streaming_attention_modules = base._streaming_attn_modules

for name, mod in streaming_attention_modules.items():
    head_dir = os.path.join(decay_dir, name)
    mod.save_pretrained(head_dir)
print(f"  Saved decay params ({len(streaming_attention_modules)} heads) to {decay_dir}")

# 2. Save LoRA adapters
lora_dir = os.path.join(SAVE_DIR, 'lora_adapters')
model.save_pretrained(lora_dir)
print(f"  Saved LoRA adapters to {lora_dir}")

# 3. Save training metadata
metadata = {
    'baseline_ppl': BASELINE_PPL,
    'stage1_ppl': stage1_ppl_result['perplexity'],
    'final_ppl': final_result['perplexity'],
    'stage1_ratio': stage1_ratio,
    'final_ratio': final_ratio,
    'streaming_fraction': STREAMING_FRACTION,
    'stage1_steps': stage1_result['num_steps'],
    'stage2_steps': stage2_result['num_steps'],
    'final_decays': stage1_result['final_decays'],
    'model_name': MODEL_NAME,
}
with open(os.path.join(SAVE_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"  Saved metadata to {SAVE_DIR}/metadata.json")

# 4. Copy loss plots
for fname in ['stage1_loss.png', 'stage2_loss.png', 'calibration_losses.png']:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(SAVE_DIR, fname))

print(f"\nCheckpoint saved to Google Drive: {SAVE_DIR}")
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Baseline PPL:      {BASELINE_PPL:.2f}")
print(f"After Stage 1:     {stage1_ppl_result['perplexity']:.2f} ({stage1_ratio:.2f}x)")
print(f"After Stage 2:     {final_result['perplexity']:.2f} ({final_ratio:.2f}x)")
print(f"Target:            < {BASELINE_PPL * 1.05:.2f}")
print(f"Status:            {'PASS' if final_ratio <= 1.05 else 'NEEDS WORK'}")